In [ ]:
%%writefile .env
DB_HOST=localhost
DB_PORT=5432
DB_NAME=TRABALHO_FDB
DB_USER=postgres
DB_PASSWORD=

Writing .env


In [3]:
import os
from dotenv import load_dotenv

import pandas as pd
import psycopg2 as pg
import sqlalchemy
from sqlalchemy import create_engine
import panel as pn

In [4]:
%pip install pandas sqlalchemy psycopg2-binary panel python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
load_dotenv(override=True)

DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASSWORD')

con = pg.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)
print("Conectado!")

Conectado!


In [6]:
cnx = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}'

engine = sqlalchemy.create_engine(cnx)

In [ ]:
query = "select * from usuario"  # Busca na tabela usuario e exibe-o
df = pd.read_sql_query(query, engine)

df

,id_usuario,email,senha,primeiro_nome,nome_meio,ultimo_nome
0,6,diogoassuncao@alu.ufc.br,123456,Diogo,Bandeira,Assunção
1,7,leandrojunior@alu.ufc.br,1234,Leandro,Rodrigues da Silva,Júnior


In [8]:
pn.extension()
pn.extension('tabulator')
pn.extension(notifications=True)

In [9]:
flag = ''

id_usuario = pn.widgets.TextInput(
    name='ID / CPF do Usuário',
    value='',
    placeholder='Digite o id_usuario ou cpf',
)
id_usuario.param.watch(lambda e: setattr(id_usuario, 'value', e.new), 'value_input')

primeiro_nome = pn.widgets.TextInput(
    name='Primeiro Nome',
    value='',
    placeholder='Digite o primeiro nome',
)
primeiro_nome.param.watch(lambda e: setattr(primeiro_nome, 'value', e.new), 'value_input')

nome_meio = pn.widgets.TextInput(
    name='Nome do Meio',
    value='',
    placeholder='Digite o nome do meio',
)
nome_meio.param.watch(lambda e: setattr(nome_meio, 'value', e.new), 'value_input')

ultimo_nome = pn.widgets.TextInput(
    name='Último Nome',
    value='',
    placeholder='Digite o último nome',
)
ultimo_nome.param.watch(lambda e: setattr(ultimo_nome, 'value', e.new), 'value_input')

email = pn.widgets.TextInput(
    name='Email',
    value='',
    placeholder='Digite o email',
)
email.param.watch(lambda e: setattr(email, 'value', e.new), 'value_input')

senha = pn.widgets.PasswordInput(
    name='Senha',
    value='',
    placeholder='Digite a senha',
)
senha.param.watch(lambda e: setattr(senha, 'value', e.new), 'value_input')

Watcher(inst=PasswordInput(label='Senha', name='Senha', placeholder='Digite a senha'), cls=<class 'panel.widgets.input.PasswordInput'>, fn=<function <lambda> at 0x112af2980>, mode='args', onlychanged=True, parameter_names=('value_input',), what='value', queued=False, precedence=0)

In [10]:
buttonConsultar = pn.widgets.Button(name='Consultar', button_type='default')

buttonInserir = pn.widgets.Button(name='Inserir', button_type='default')

buttonExcluir = pn.widgets.Button(name='Excluir', button_type='default')

buttonAtualizar = pn.widgets.Button(name='Atualizar', button_type='default')

In [11]:
def queryAll():
    """
    Consulta todos os registros da tabela 'Usuario' no banco de dados e retorna
    um widget Tabulator para exibição interativa dos dados.

    Returns:
        pn.widgets.Tabulator: Widget que exibe a tabela com todos os dados da tabela 'Usuario'.
    """
    query = "select * from usuario"
    df = pd.read_sql_query(query, engine)
    return pn.widgets.Tabulator(df)


# consultar
# se id_usuario estiver vazio, retorna todos os registros.
# se tiver algo digitado, filtra pela coluna id_usuario (chave primária da tabela).
def on_consultar():
    """
    Consulta registros na tabela 'usuario' filtrando pelo id_usuario informado.
    Se o campo estiver vazio, retorna todos os registros.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela com os dados encontrados ou alerta em caso de erro.
    """
    try:
        valor = id_usuario.value_input.strip()
        if valor == '':
            df = pd.read_sql_query("select * from usuario", engine)
        else:
            df = pd.read_sql_query(
                "select * from usuario where id_usuario = %s",
                engine,
                params=(valor,)
            )
        return pn.widgets.Tabulator(df)
    except Exception as e:
        return pn.pane.Alert(f'Não foi possível consultar: {str(e)}')


def on_inserir():
    try:            
        cursor = con.cursor()
        # Padronizando tudo para .value para capturar os textos perfeitamente
        cursor.execute("insert into usuario(primeiro_nome, nome_meio, ultimo_nome, email, senha) VALUES (%s, %s, %s, %s, %s)", 
                    (primeiro_nome.value, nome_meio.value, ultimo_nome.value, email.value, senha.value))
        con.commit()
        cursor.close()
        return queryAll()
    except Exception as e:
        # No psycopg2, o rollback deve ser feito direto na conexão (con), não no cursor
        con.rollback()  
        if 'cursor' in locals():
            cursor.close()
        return pn.pane.Alert(f'Não foi possível inserir: {str(e)}')


def on_atualizar():
    """
    Atualiza os campos email e senha do registro identificado pelo id informado.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        cursor= con.cursor()
        cursor.execute("UPDATE usuario SET email = %s, senha = %s WHERE id_usuario = %s",
           (email.value_input, senha.value, id_usuario.value_input))
        cursor.query
        cursor.query
        con.commit()
        return queryAll()
    except Exception as e:
        cursor.execute("ROLLBACK")
        cursor.close()
        return pn.pane.Alert(f'Não foi possível atualizar: {str(e)}')


def on_excluir():
    """
    Exclui o registro da tabela 'usuario' com o id_usuario informado.

    Returns:
        pn.widgets.Tabulator ou pn.pane.Alert: Tabela atualizada ou alerta em caso de erro.
    """
    try:
        cursor= con.cursor()
        cursor.execute("delete from usuario where id_usuario=%s", (id_usuario.value_input,))
        rows_deleted = cursor.rowcount
        con.commit()
        return queryAll()
    except:
        cursor.execute("ROLLBACK")            
        cursor.close() 
        return pn.pane.Alert('Não foi possível excluir!')

In [12]:
def table_creator(cons, ins, atu, exc):
    if cons:
        return on_consultar()
    if ins:
        return on_inserir()
    if atu:
        return on_atualizar()
    if exc:
        return on_excluir()

In [13]:
def tela_usuario():
    """Tela de CRUD completo da entidade Usuario."""
    interactive_table = pn.bind(table_creator, buttonConsultar, buttonInserir, buttonAtualizar, buttonExcluir)
    return pn.Row(
        pn.Column(
            '## CRUD - Usuário',
            '#### Buscar / Atualizar / Excluir (informe o ID)',
            id_usuario,
            pn.Row(buttonConsultar, buttonAtualizar, buttonExcluir),
            pn.layout.Divider(),
            '#### Novo usuário (Inserir)',
            primeiro_nome, nome_meio, ultimo_nome, email, senha,
            pn.Row(buttonInserir)
        ),
        pn.Column(interactive_table)
    )

In [14]:
# Chamada de teste antiga - agora o Inserir é feito pela tela (botão), não precisa mais chamar direto.
# on_inserir()

## CRUD - Aluno (segunda entidade)

A chave de Aluno é o mesmo `id_usuario` do Usuário (relação de subtipo: 1 Usuário pode ser 1 Aluno). Pra inserir um Aluno, informe o `id_usuario` de um Usuário que já existe na tabela Usuario.

In [15]:
id_aluno = pn.widgets.TextInput(
    name='ID do Usuário (Aluno)',
    value='',
    placeholder='id_usuario do Usuário já cadastrado',
)
id_aluno.param.watch(lambda e: setattr(id_aluno, 'value', e.new), 'value_input')

matricula_aluno = pn.widgets.TextInput(
    name='Matrícula',
    value='',
    placeholder='Digite a matrícula',
)
matricula_aluno.param.watch(lambda e: setattr(matricula_aluno, 'value', e.new), 'value_input')

curso_aluno = pn.widgets.TextInput(
    name='Curso',
    value='',
    placeholder='Digite o curso',
)
curso_aluno.param.watch(lambda e: setattr(curso_aluno, 'value', e.new), 'value_input')

semestre_aluno = pn.widgets.IntInput(
    name='Semestre',
    value=1,
    start=1,
    end=20,
)

buttonConsultarAluno = pn.widgets.Button(name='Consultar', button_type='default')
buttonInserirAluno = pn.widgets.Button(name='Inserir', button_type='default')
buttonAtualizarAluno = pn.widgets.Button(name='Atualizar', button_type='default')
buttonExcluirAluno = pn.widgets.Button(name='Excluir', button_type='default')

In [16]:
def queryAllAluno():
    df = pd.read_sql_query("select * from aluno", engine)
    return pn.widgets.Tabulator(df)


def on_consultar_aluno():
    try:
        valor = id_aluno.value_input.strip()
        if valor == '':
            df = pd.read_sql_query("select * from aluno", engine)
        else:
            df = pd.read_sql_query(
                "select * from aluno where id_usuario = %s",
                engine, params=(valor,)
            )
        return pn.widgets.Tabulator(df)
    except Exception as e:
        return pn.pane.Alert(f'Não foi possível consultar: {str(e)}')


def on_inserir_aluno():
    if id_aluno.value_input.strip() == '' or matricula_aluno.value.strip() == '' or curso_aluno.value.strip() == '':
        return pn.pane.Alert('Preencha ID do Usuário, Matrícula e Curso antes de inserir.')
    try:
        cursor = con.cursor()
        cursor.execute(
            "insert into aluno(id_usuario, matricula, curso, semestre) VALUES (%s, %s, %s, %s)",
            (id_aluno.value, matricula_aluno.value, curso_aluno.value, semestre_aluno.value)
        )
        con.commit()
        cursor.close()
        return queryAllAluno()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Não foi possível inserir: {str(e)}')


def on_atualizar_aluno():
    if matricula_aluno.value.strip() == '' or curso_aluno.value.strip() == '':
        return pn.pane.Alert('Preencha Matrícula e Curso antes de atualizar.')
    try:
        cursor = con.cursor()
        cursor.execute(
            "UPDATE aluno SET matricula = %s, curso = %s, semestre = %s WHERE id_usuario = %s",
            (matricula_aluno.value, curso_aluno.value, semestre_aluno.value, id_aluno.value_input)
        )
        con.commit()
        cursor.close()
        return queryAllAluno()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Não foi possível atualizar: {str(e)}')


def on_excluir_aluno():
    try:
        cursor = con.cursor()
        cursor.execute("delete from aluno where id_usuario = %s", (id_aluno.value_input,))
        con.commit()
        cursor.close()
        return queryAllAluno()
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(f'Não foi possível excluir: {str(e)}')


def table_creator_aluno(cons, ins, atu, exc):
    if cons:
        return on_consultar_aluno()
    if ins:
        return on_inserir_aluno()
    if atu:
        return on_atualizar_aluno()
    if exc:
        return on_excluir_aluno()

In [17]:
def tela_aluno():
    """Tela de CRUD completo da entidade Aluno."""
    interactive_table_aluno = pn.bind(
        table_creator_aluno, buttonConsultarAluno, buttonInserirAluno, buttonAtualizarAluno, buttonExcluirAluno
    )
    return pn.Row(
        pn.Column(
            '## CRUD - Aluno',
            '#### Buscar / Atualizar / Excluir (informe o ID do Usuário)',
            id_aluno,
            pn.Row(buttonConsultarAluno, buttonAtualizarAluno, buttonExcluirAluno),
            pn.layout.Divider(),
            '#### Novo aluno (Inserir) - use o ID de um Usuário já existente',
            matricula_aluno, curso_aluno, semestre_aluno,
            pn.Row(buttonInserirAluno)
        ),
        pn.Column(interactive_table_aluno)
    )

## Relatório

Lista todos os alunos (join Usuário + Aluno) e um resumo com total de alunos e média de semestre por curso. Clique em **Atualizar Relatório** para recarregar com os dados mais recentes.

In [18]:
buttonAtualizarRelatorio = pn.widgets.Button(name='Atualizar Relatório', button_type='primary')


def gerar_lista_alunos():
    query = """
        select u.id_usuario, u.primeiro_nome, u.nome_meio, u.ultimo_nome, u.email,
               a.matricula, a.curso, a.semestre
        from usuario u
        join aluno a on a.id_usuario = u.id_usuario
        order by u.primeiro_nome
    """
    df = pd.read_sql_query(query, engine)
    return pn.widgets.Tabulator(df, disabled=True, sizing_mode='stretch_width')


def gerar_resumo_por_curso():
    query = """
        select curso,
               count(*) as total_alunos,
               round(avg(semestre), 1) as media_semestre
        from aluno
        group by curso
        order by total_alunos desc
    """
    df = pd.read_sql_query(query, engine)
    return pn.widgets.Tabulator(df, disabled=True, sizing_mode='stretch_width')


def conteudo_relatorio(clicked):
    try:
        return pn.Column(
            '#### Lista de alunos (Usuário + Aluno)',
            gerar_lista_alunos(),
            pn.layout.Divider(),
            '#### Resumo por curso',
            gerar_resumo_por_curso(),
        )
    except Exception as e:
        return pn.pane.Alert(f'Não foi possível gerar o relatório: {str(e)}')


def tela_relatorio():
    """Tela de relatório: lista de alunos e resumo agregado por curso."""
    relatorio_interativo = pn.bind(conteudo_relatorio, buttonAtualizarRelatorio)
    return pn.Column(
        '## Relatório de Alunos',
        buttonAtualizarRelatorio,
        relatorio_interativo
    )

## Menu Principal

Navegação entre as telas do sistema.

In [19]:
def tela_menu():
    btn_usuario = pn.widgets.Button(name='Usuário (CRUD)', button_type='primary')
    btn_aluno = pn.widgets.Button(name='Aluno (CRUD)', button_type='primary')
    btn_relatorio = pn.widgets.Button(name='Relatório', button_type='primary')

    def ir_para_usuario(event):
        menu_principal.active = 1

    def ir_para_aluno(event):
        menu_principal.active = 2

    def ir_para_relatorio(event):
        menu_principal.active = 3

    btn_usuario.on_click(ir_para_usuario)
    btn_aluno.on_click(ir_para_aluno)
    btn_relatorio.on_click(ir_para_relatorio)

    return pn.Column(
        '# Sistema de Gestão',
        'Trabalho de Fundamentos de Banco de Dados',
        pn.Spacer(height=20),
        btn_usuario,
        btn_aluno,
        btn_relatorio,
    )


menu_principal = pn.Tabs(
    ('Menu', tela_menu()),
    ('Usuário', tela_usuario()),
    ('Aluno', tela_aluno()),
    ('Relatório', tela_relatorio()),
)
menu_principal.servable()

Tabs
    [0] Column
        [0] Markdown(str)
        [1] Markdown(str)
        [2] Spacer(height=20)
        [3] Button(button_type='primary', color='primary', label='Usuário (CRUD)', name='Usuário (CRUD)')
        [4] Button(button_type='primary', color='primary', label='Aluno (CRUD)', name='Aluno (CRUD)')
        [5] Button(button_type='primary', color='primary', label='Relatório', name='Relatório')
    [1] Row
        [0] Column
            [0] Markdown(str)
            [1] Markdown(str)
            [2] TextInput(label='ID / CPF do Usuário', name='ID / CPF do Usuário', placeholder='Digite o id_usuario o...)
            [3] Row
                [0] Button(label='Consultar', name='Consultar')
                [1] Button(label='Atualizar', name='Atualizar')
                [2] Button(label='Excluir', name='Excluir')
            [4] Divider()
            [5] Markdown(str)
            [6] TextInput(label='Primeiro Nome', name='Primeiro Nome', placeholder='Digite o primeiro nome')
            [7] TextInput(label='Nome do Meio', name='Nome do Meio', placeholder='Digite o nome do meio')
            [8] TextInput(label='Último Nome', name='Último Nome', placeholder='Digite o último nome')
            [9] TextInput(label='Email', name='Email', placeholder='Digite o email')
            [10] PasswordInput(label='Senha', name='Senha', placeholder='Digite a senha')
            [11] Row
                [0] Button(label='Inserir', name='Inserir')
        [1] Column
            [0] ParamFunction(function, _pane=Str, defer_load=False)
    [2] Row
        [0] Column
            [0] Markdown(str)
            [1] Markdown(str)
            [2] TextInput(label='ID do Usuário (Aluno)', name='ID do Usuário (Aluno)', placeholder='id_usuario do U...)
            [3] Row
                [0] Button(label='Consultar', name='Consultar')
                [1] Button(label='Atualizar', name='Atualizar')
                [2] Button(label='Excluir', name='Excluir')
            [4] Divider()
            [5] Markdown(str)
            [6] TextInput(label='Matrícula', name='Matrícula', placeholder='Digite a matrícula')
            [7] TextInput(label='Curso', name='Curso', placeholder='Digite o curso')
            [8] IntInput(end=20, label='Semestre', name='Semestre', start=1, value=1)
            [9] Row
                [0] Button(label='Inserir', name='Inserir')
        [1] Column
            [0] ParamFunction(function, _pane=Str, defer_load=False)
    [3] Column
        [0] Markdown(str)
        [1] Button(button_type='primary', color='primary', label='Atualizar Relatório', name='Atualizar Relatório')
        [2] ParamFunction(function, _pane=Column, defer_load=False)